In [1]:
import trimesh
import open3d as o3d
import numpy as np
import os

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ==========================
# CARGAR MODELO
# ==========================
def load_mesh(path):
    print(f"\nCargando: {path}")
    mesh = trimesh.load(path)
    
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(mesh.dump())
    
    return mesh


In [3]:
# ==========================
# ANALIZAR PROPIEDADES
# ==========================
def analyze_mesh(mesh, name="Mesh"):
    print(f"\n--- Análisis de {name} ---")
    
    vertices = len(mesh.vertices)
    faces = len(mesh.faces)
    normals = len(mesh.vertex_normals) if mesh.vertex_normals is not None else 0
    
    print(f"Vértices: {vertices}")
    print(f"Caras: {faces}")
    print(f"Normales: {normals}")
    
    # Detectar vértices duplicados
    unique_vertices = len(np.unique(mesh.vertices, axis=0))
    duplicated = vertices - unique_vertices
    
    print(f"Vértices duplicados: {duplicated}")
    
    print(f"¿Es watertight (cerrado)? {mesh.is_watertight}")
    print(f"Área superficial: {mesh.area}")
    print(f"Volumen: {mesh.volume if mesh.is_watertight else 'No aplica'}")


In [4]:
# ==========================
# VISUALIZAR CON OPEN3D
# ==========================
def visualize_mesh(mesh):
    mesh_o3d = o3d.geometry.TriangleMesh()
    
    mesh_o3d.vertices = o3d.utility.Vector3dVector(mesh.vertices)
    mesh_o3d.triangles = o3d.utility.Vector3iVector(mesh.faces)
    
    mesh_o3d.compute_vertex_normals()
    
    o3d.visualization.draw_geometries([mesh_o3d])


In [5]:
# ==========================
# CONVERTIR FORMATO
# ==========================
def convert_mesh(input_path, output_path):
    mesh = load_mesh(input_path)
    mesh.export(output_path)
    print(f"Convertido y guardado en: {output_path}")

In [6]:
# ==========================
# COMPARAR DOS MODELOS
# ==========================
def compare_meshes(mesh1, mesh2, name1="Mesh1", name2="Mesh2"):
    print("\n========== COMPARACIÓN ==========")
    
    print(f"\nComparando {name1} vs {name2}")
    
    print(f"Diferencia en vértices: {len(mesh1.vertices) - len(mesh2.vertices)}")
    print(f"Diferencia en caras: {len(mesh1.faces) - len(mesh2.faces)}")
    print(f"Diferencia en área: {mesh1.area - mesh2.area}")

In [7]:
# ==========================
# AUTOMATIZAR COMPARACION
# ==========================
def analyze_folder(folder_path):
    meshes = {}
    
    for file in os.listdir(folder_path):
        if file.endswith((".obj", ".stl", ".gltf", ".glb")):
            path = os.path.join(folder_path, file)
            mesh = load_mesh(path)
            meshes[file] = mesh
            analyze_mesh(mesh, file)
    
    # Comparación automática
    files = list(meshes.keys())
    for i in range(len(files)):
        for j in range(i+1, len(files)):
            compare_meshes(meshes[files[i]], meshes[files[j]], files[i], files[j])

In [9]:
# ==========================
# USO
# ==========================
if __name__ == "__main__":

    # Cargar dos modelos
    mesh1 = load_mesh("modelos/chair_gltf.gltf")
    mesh2 = load_mesh("modelos/chair_obj.obj")
    mesh3 = load_mesh("modelos/chair_stl.stl")
    
    """
    analyze_mesh(mesh1, "Modelo 1")
    analyze_mesh(mesh2, "Modelo 2")
    analyze_mesh(mesh3, "Modelo 3")
   
    compare_meshes(mesh1, mesh2, mesh3)
    """
    
    analyze_folder("modelos")
    
    # Visualizar cada uno
    visualize_mesh(mesh1)
    visualize_mesh(mesh2)
    visualize_mesh(mesh3)
    
    
    # Convertir formato
    convert_mesh("modelos/chair_gltf.gltf", "output/chair_gltf.stl")
    convert_mesh("modelos/chair_obj.obj", "output/chair_obj.gltf")
    convert_mesh("modelos/chair_stl.stl", "output/chair_stl.obj")

    
    # Cargar modelos exportados
    mesh4 = load_mesh("output/chair_gltf.stl")
    mesh5 = load_mesh("output/chair_obj.gltf")
    mesh6 = load_mesh("output/chair_stl.obj")

    # Visualizar cada uno
    visualize_mesh(mesh4)
    visualize_mesh(mesh5)
    visualize_mesh(mesh6)
    


Cargando: modelos/chair_gltf.gltf

Cargando: modelos/chair_obj.obj

Cargando: modelos/chair_stl.stl

Cargando: modelos\chair_gltf.gltf

--- Análisis de chair_gltf.gltf ---
Vértices: 26730
Caras: 43392
Normales: 26730
Vértices duplicados: 5024
¿Es watertight (cerrado)? False
Área superficial: 1.5667466530663183
Volumen: No aplica

Cargando: modelos\chair_obj.obj

--- Análisis de chair_obj.obj ---
Vértices: 44734
Caras: 86552
Normales: 44734
Vértices duplicados: 0
¿Es watertight (cerrado)? False
Área superficial: 1164188.993817733
Volumen: No aplica

Cargando: modelos\chair_stl.stl

--- Análisis de chair_stl.stl ---
Vértices: 7937
Caras: 15840
Normales: 7937
Vértices duplicados: 0
¿Es watertight (cerrado)? False
Área superficial: 1324317.8811091688
Volumen: No aplica

========== COMPARACIÓN ==========

Comparando chair_gltf.gltf vs chair_obj.obj
Diferencia en vértices: -18004
Diferencia en caras: -43160
Diferencia en área: -1164187.4270710798

========== COMPARACIÓN ==========

Comparan